In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import scipy.stats as stats
import statsmodels.formula.api as smf
from pathlib import Path

DATA_DIR = Path('/Users/jackzipper/QSS20/final_project/final_project_data')
OUT_DIR  = Path('/Users/jackzipper/QSS20/final_project/output')
OUT_DIR.mkdir(exist_ok=True)

# Table layout constants
BG     = 'white'
FG     = 'black'
ACCENT = '#444444'
RULE   = '#aaaaaa'
COLS   = [0.04, 0.40, 0.68]
Y0     = 0.91
ROW_H  = 0.090


# ── Helper functions ───────────────────────────────────────────────────────

def stars(p):
    """Significance stars: *** p<0.01, ** p<0.05, * p<0.1."""
    if p < 0.01: return '***'
    if p < 0.05: return '**'
    if p < 0.10: return '*'
    return ''


def row(ax, y, cells, bold=False, color=FG, small=False):
    """Render a single row of monospaced text across COLS positions."""
    fs = 9.5 if not small else 8.5
    fw = 'bold' if bold else 'normal'
    for x, txt in zip(COLS, cells):
        ax.text(x, y, txt, transform=ax.transAxes,
                color=color, fontsize=fs, fontweight=fw,
                va='top', ha='left', fontfamily='monospace')


def hline(ax, y, lw=0.8, color=RULE):
    """Draw a horizontal rule spanning the table width."""
    ax.plot([0.03, 0.97], [y, y], color=color, linewidth=lw,
            transform=ax.transAxes, clip_on=False)


In [2]:
# ── Load quarterly metric computed by 07_choropleth_map ───────────────────
# Death-fill + aggregation logic lives in 07. Edit it there; 08 reads the output.
qdf = pd.read_csv(DATA_DIR / 'drc_quarterly_metric.csv')
qdf['quarter'] = pd.PeriodIndex(qdf['quarter'], freq='Q')

print(f'{len(qdf):,} territory-quarter observations')
print(f'metric range [{qdf["metric"].min():.2f}, {qdf["metric"].max():.2f}]')


3,504 territory-quarter observations
metric range [-5.32, 19.15]


In [3]:
# ── Define under-served ────────────────────────────────────────────────────
# Global threshold = bottom quartile of metric across ALL territory-quarters.
# This is a fixed baseline so counts are comparable across time.
THRESHOLD = qdf['metric'].quantile(0.25)
print(f'Under-served threshold (25th pct): {THRESHOLD:.3f}')

qdf['underserved'] = (qdf['metric'] <= THRESHOLD).astype(int)

# Count per quarter
ts = (
    qdf.groupby('quarter')
    .agg(
        n_underserved  = ('underserved', 'sum'),
        n_total        = ('underserved', 'count'),
        pct_underserved= ('underserved', 'mean'),
    )
    .reset_index()
)
ts['quarter_dt'] = ts['quarter'].dt.to_timestamp()
ts['t'] = np.arange(len(ts))  # integer time index for regression

print(ts[['quarter','n_underserved','pct_underserved']].to_string())

Under-served threshold (25th pct): 10.397
   quarter  n_underserved  pct_underserved
0   2021Q1             26         0.178082
1   2021Q2             29         0.198630
2   2021Q3             27         0.184932
3   2021Q4             31         0.212329
4   2022Q1             33         0.226027
5   2022Q2             29         0.198630
6   2022Q3             27         0.184932
7   2022Q4             26         0.178082
8   2023Q1             27         0.184932
9   2023Q2             27         0.184932
10  2023Q3             25         0.171233
11  2023Q4             26         0.178082
12  2024Q1             23         0.157534
13  2024Q2             27         0.184932
14  2024Q3             32         0.219178
15  2024Q4             32         0.219178
16  2025Q1             48         0.328767
17  2025Q2             52         0.356164
18  2025Q3             50         0.342466
19  2025Q4             52         0.356164
20  2026Q1             55         0.376712
21  2026Q2  

In [4]:
# ── OLS trend test ─────────────────────────────────────────────────────────
model_count = smf.ols('n_underserved ~ t', data=ts).fit()
model_pct   = smf.ols('pct_underserved ~ t', data=ts).fit()

print('=== Count of under-served territories ~ time ===')
print(model_count.summary2().tables[1])
print()
print('=== % under-served ~ time ===')
print(model_pct.summary2().tables[1])

=== Count of under-served territories ~ time ===
               Coef.  Std.Err.         t         P>|t|     [0.025     0.975]
Intercept  19.510000  2.952243  6.608535  1.205260e-06  13.387423  25.632577
t           1.477391  0.219945  6.717082  9.438997e-07   1.021252   1.933530

=== % under-served ~ time ===
              Coef.  Std.Err.         t         P>|t|    [0.025    0.975]
Intercept  0.133630  0.020221  6.608535  1.205260e-06  0.091695  0.175566
t          0.010119  0.001506  6.717082  9.438997e-07  0.006995  0.013243


In [5]:
# ── Mann-Kendall monotonic trend test (non-parametric) ─────────────────────
# Tests whether the series is monotonically increasing/decreasing without
# assuming normality.
mk_count = stats.kendalltau(ts['t'], ts['n_underserved'])
mk_pct   = stats.kendalltau(ts['t'], ts['pct_underserved'])

print(f'Mann-Kendall (count):  tau={mk_count.statistic:.3f}, p={mk_count.pvalue:.4f}')
print(f'Mann-Kendall (pct  ):  tau={mk_pct.statistic:.3f},   p={mk_pct.pvalue:.4f}')

Mann-Kendall (count):  tau=0.535, p=0.0004
Mann-Kendall (pct  ):  tau=0.535,   p=0.0004


In [6]:
# ── Plot: % of territories under-served — two-period slopes ───────────────
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
BG = 'white'

# Variables used by the interpretation cell below
p_count     = model_count.pvalues['t']
slope_count = model_count.params['t']
p_pct       = model_pct.pvalues['t']
slope_pct   = model_pct.params['t']

# ── Breakpoint: 2025Q1 (Trump inauguration / USAID freeze) ─────────────────
BREAK_T    = 16
break_date = ts.loc[ts['t'] == BREAK_T, 'quarter_dt'].values[0]

ts_pre  = ts[ts['t'] <  BREAK_T]
ts_post = ts[ts['t'] >= BREAK_T]

# Fit separate OLS lines on each segment
m_pre  = smf.ols('pct_underserved ~ t', data=ts_pre).fit()
m_post = smf.ols('pct_underserved ~ t', data=ts_post).fit()

# ── Build trend lines that meet exactly at break_date ──────────────────────
# Quarters are uniformly spaced; used to convert dates → fractional t
HALF_BAR   = pd.Timedelta(days=30)
days_per_t = (ts['quarter_dt'].iloc[1] - ts['quarter_dt'].iloc[0]).days  # ~91

def date_to_t(dates, ref_date, ref_t, days_per_t):
    return ref_t + (dates - ref_date).days / days_per_t

def eval_trend(model, t_vals, scale=100):
    return (model.params['Intercept'] + model.params['t'] * t_vals) * scale

# Pre-break: left edge of first bar → break_date
pre_x = pd.date_range(ts_pre['quarter_dt'].iloc[0] - HALF_BAR, break_date, periods=200)
pre_t = date_to_t(pre_x, ts_pre['quarter_dt'].iloc[0], ts_pre['t'].iloc[0], days_per_t)
pre_y = eval_trend(m_pre, pre_t)

# Post-break: break_date → right edge of last bar
post_x = pd.date_range(break_date, ts_post['quarter_dt'].iloc[-1] + HALF_BAR, periods=200)
post_t = date_to_t(post_x, ts_post['quarter_dt'].iloc[0], ts_post['t'].iloc[0], days_per_t)
post_y = eval_trend(m_post, post_t)

# ── Draw ───────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)
ax.spines[['top', 'right']].set_visible(False)
ax.spines[['bottom', 'left']].set_color('#444444')
ax.tick_params(colors='black')

ax.bar(ts['quarter_dt'], ts['pct_underserved'] * 100, width=60,
       color='#cb181d', alpha=0.80, label='% under-served')

# Pre-break trend (dark/black) — ends at break_date
ax.plot(pre_x, pre_y, color='#222222', linewidth=2, linestyle='--',
        label=f'Pre-break trend ({m_pre.params["t"]*100:+.2f}%/qtr, p={m_pre.pvalues["t"]:.3f})')

# Post-break trend (gold) — starts at break_date
ax.plot(post_x, post_y, color='#d4a800', linewidth=2.5, linestyle='--',
        label=f'Post-break trend ({m_post.params["t"]*100:+.2f}%/qtr, p={m_post.pvalues["t"]:.3f})')

# Vertical break line
ax.axvline(break_date, color='#444444', linewidth=1.2, linestyle=':', zorder=5)

# Annotation — Trump inauguration / USAID freeze
ax.annotate(
    'Trump inauguration\n& USAID freeze\n(Jan 2025)',
    xy=(break_date, ax.get_ylim()[1] * 0.97),
    xytext=(break_date - pd.Timedelta(days=280), ax.get_ylim()[1] * 0.88),
    arrowprops=dict(arrowstyle='->', color='#444444', lw=1.1),
    fontsize=8.5, color='#444444', ha='right', va='top',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='#cccccc', alpha=0.9)
)

ax.set_title('Under-Served Territories Over Time',
             color='black', fontsize=13, fontweight='bold', pad=10)
ax.set_ylabel('% of territories under-served', color='black', fontsize=11)
ax.set_xlabel('Quarter', color='black', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))

ax.legend(facecolor='white', labelcolor='black', fontsize=8.5,
          edgecolor='#cccccc', loc='upper left')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()

out_path = OUT_DIR / 'underserved_trend.png'
fig.savefig(str(out_path), dpi=150, bbox_inches='tight', facecolor=BG)
plt.close(fig)
print(f'Plot saved → {out_path}')
print(f'\nPre-break  slope: {m_pre.params["t"]*100:+.3f}%/qtr  (p={m_pre.pvalues["t"]:.4f})')
print(f'Post-break slope: {m_post.params["t"]*100:+.3f}%/qtr (p={m_post.pvalues["t"]:.4f})')


Plot saved → /Users/jackzipper/QSS20/final_project/output/underserved_trend.png

Pre-break  slope: -0.011%/qtr  (p=0.9199)
Post-break slope: +0.987%/qtr (p=0.0002)


In [7]:
# ── Interpretation ─────────────────────────────────────────────────────────
alpha = 0.05
sig = p_count < alpha
direction = 'increased' if slope_count > 0 else 'decreased'

print('\n=== SUMMARY ===')
print(f'Under-served threshold: {THRESHOLD:.2f} (bottom-quartile log($/death))')
print(f'OLS: under-served count has {direction} by {abs(slope_count):.2f} territories/quarter')
print(f'     p-value = {p_count:.4f} → {"statistically significant" if sig else "NOT significant"} at alpha=0.05')
print(f'Mann-Kendall: tau = {mk_count.statistic:.3f}, p = {mk_count.pvalue:.4f}')


=== SUMMARY ===
Under-served threshold: 10.40 (bottom-quartile log($/death))
OLS: under-served count has increased by 1.48 territories/quarter
     p-value = 0.0000 → statistically significant at alpha=0.05
Mann-Kendall: tau = 0.535, p = 0.0004


In [ ]:
# ── Regression table as PNG ────────────────────────────────────────────────
# stars(), row(), hline() defined in the top imports cell
m1, m2 = model_count, model_pct

fig_t, ax_t = plt.subplots(figsize=(10, 5.4))
fig_t.patch.set_facecolor(BG)
ax_t.set_facecolor(BG)
ax_t.set_axis_off()
ax_t.set_xlim(0, 1)
ax_t.set_ylim(0, 1)

# Title
ax_t.text(0.5, 0.98,
    'OLS Regressions: Under-Served Territories ~ Quarter Index (t)',
    transform=ax_t.transAxes, color=FG, fontsize=11.5,
    fontweight='bold', ha='center', va='top')

y = Y0
hline(ax_t, y, lw=1.2, color='black')
y -= 0.01

# Column headers
row(ax_t, y, ['', 'Model 1: Count', 'Model 2: Share (ppts)'], bold=True, color=ACCENT)
y -= ROW_H
hline(ax_t, y, lw=0.6)
y -= 0.01

# Intercept
row(ax_t, y, [
    'Intercept',
    f'{m1.params["Intercept"]:8.3f}{stars(m1.pvalues["Intercept"])}',
    f'{m2.params["Intercept"]*100:8.3f}{stars(m2.pvalues["Intercept"])}',
])
y -= ROW_H * 0.68
row(ax_t, y, ['',
    f'({m1.bse["Intercept"]:.3f})',
    f'({m2.bse["Intercept"]*100:.3f})',
], color=ACCENT, small=True)
y -= ROW_H * 0.90

# Slope
row(ax_t, y, [
    'Quarter index (t)',
    f'{m1.params["t"]:8.3f}{stars(m1.pvalues["t"])}',
    f'{m2.params["t"]*100:8.4f}{stars(m2.pvalues["t"])}',
])
y -= ROW_H * 0.68
row(ax_t, y, ['',
    f'({m1.bse["t"]:.3f})',
    f'({m2.bse["t"]*100:.4f})',
], color=ACCENT, small=True)
y -= ROW_H * 0.90

hline(ax_t, y, lw=0.6)
y -= 0.01

# Fit statistics
row(ax_t, y, ['N', str(int(m1.nobs)), str(int(m2.nobs))])
y -= ROW_H * 0.80
row(ax_t, y, ['R²', f'{m1.rsquared:.3f}', f'{m2.rsquared:.3f}'])
y -= ROW_H * 0.80
row(ax_t, y, [
    'Mann-Kendall τ',
    f'{mk_count.statistic:.3f}  (p={mk_count.pvalue:.4f})',
    f'{mk_pct.statistic:.3f}  (p={mk_pct.pvalue:.4f})',
])
y -= ROW_H * 0.80

hline(ax_t, y, lw=1.2, color='black')
y -= 0.01

# Footnote
ax_t.text(0.04, y,
    'Standard errors in parentheses.  *** p<0.01  ** p<0.05  * p<0.1\n'
    'Dependent variable: # / share of territories in bottom quartile of log($/death) per quarter.\n'
    'Model 2 coefficients and SEs expressed in percentage points (×100).',
    transform=ax_t.transAxes, color=ACCENT, fontsize=8.0,
    va='top', fontfamily='monospace')

reg_path = OUT_DIR / 'underserved_regression.png'
fig_t.savefig(str(reg_path), dpi=150, bbox_inches='tight', facecolor=BG)
plt.close(fig_t)
print(f'Regression table saved → {reg_path}')
